In [ ]:
# This code compares fitness across different selection and replacement strategies
# (rank-based selection with different bias, tournament selection with different k parameters,
# mu comma lambda, and mu plus lambda replacement strategies)
# penalized fitness without novelty component is used for all the strategies
from functools import partial
from src.controller.ga import (
    GeneticAlgorithm, GAConfig,
    tournament_selection, rank_selection,
    mu_plus_lambda, mu_comma_lambda
)
from src.model.population import Population
from src.model.molecule import Molecule
from src.view.plots import plot_multiple_fitness_histories
from src.model.fitness import compute_fitness_penalized  # or your compute_penalized_fitness
from src.model.fitness import archive

# initial molecules
soup = ['[C][#N]', '[C][=O]', '[C][O]', '[C][C][O]', '[C][C][=O]',
         '[O][=C][C][O]', '[O][=C][O]', '[N][C][=Branch1][C][=O][N]', 
         '[N]', '[O]', '[N][C][C][=Branch1][C][=O][O]', '[C][C][Branch1][=Branch1][C][=Branch1][C][=O][O][N]', 
         '[C][C][=Branch1][C][=O][O]', '[C][C][N]', '[C][S]', '[C][C][=Branch1][C][=O][C][=Branch1][C][=O][O]', 
         '[C][C][=Branch1][C][=O][C]', '[O][=C][=O]', '[O][=C][=S]', '[O][P][=Branch1][C][=O][Branch1][C][O][O]', 
         '[C][=C][C][=C][C][=C][Ring1][=Branch1]', '[C][=C][N][=C][NH1][Ring1][Branch1]', '[C][C][=C][NH1][C][=Ring1][Branch1]', 
         '[C][C][C][C][C][Ring1][Branch1]', '[C][C][C][C][C][C][Ring1][=Branch1]', '[N][C][=N][C][=C][N][Ring1][Branch1]', 
         '[C][C][=C][O][C][=Ring1][Branch1]', '[O][C][C][=Branch1][C][=O][O]', '[C][=N][C][=N][C][NH1][C][=N][C][Ring1][=Branch2][=Ring1][Branch1]', 
         '[O][P][=Branch1][C][=O][Branch1][C][O][O][P][=Branch1][C][=O][Branch1][C][O][O]', '[C][C][Branch1][C][O][C][=Branch1][C][=O][O]', 
         '[O][=C][C][Branch1][C][O][C][O]', '[O][=C][Branch1][Ring1][C][O][C][O]', '[N][C][=O]', '[C][=C]', '[C][C][C][=Branch1][C][=O][O]', 
         '[O][=C][Branch1][C][O][C][C][C][=Branch1][C][=O][O]', '[N][C][C][S]', '[N][C][=Branch1][C][=S][N]', 
         '[O][C][C@H1][O][C][Branch1][C][O][C@H1][Branch1][C][O][C@@H1][Ring1][#Branch1][O]']

initial = [Molecule(s) for s in soup]

def make_initial_population():
    # fresh population for every run
    return Population(initial)

# single penalized fitness function for all runs
fitness_fn = compute_fitness_penalized

# creating a parent selector function that binds selection hyperparameters (k or bias)
def make_parent_selector_tournament(k):
    return lambda population: tournament_selection(population.molecules, population.fitness, k)

def make_parent_selector_rank(bias):
    return lambda population: rank_selection(population.molecules, population.fitness, bias=bias)

strategy_runs = [
    ("Tournament(k=2) + (μ+λ)", make_parent_selector_tournament(k=2), mu_plus_lambda),
    ("Tournament(k=4) + (μ+λ)", make_parent_selector_tournament(k=4), mu_plus_lambda),
    ("Rank(bias=1.3) + (μ+λ)", make_parent_selector_rank(bias=1.3), mu_plus_lambda),
    ("Rank(bias=2.0) + (μ+λ)", make_parent_selector_rank(bias=2.0), mu_plus_lambda),
    ("Tournament(k=2) + (μ,λ)", make_parent_selector_tournament(k=2), mu_comma_lambda),
    ("Rank(bias=1.3) + (μ,λ)", make_parent_selector_rank(bias=1.3), mu_comma_lambda),
]

histories = []
labels = []

for label, parent_selector, replacer in strategy_runs:
    # reset the archive memory
    archive.archive.clear()

    print(f"\n=== Running GA with {label} ===")
    pop = make_initial_population()  

    cfg = GAConfig(
        mu=100,
        lam=100,
        mutation_rate=1,
        crossover_rate=1,
        tournament_k=2,   # only used if not set in parent_selector
        rank_bias=1.7,    # only used if not set in parent_selector
        random_seed=0,
    )

    ga = GeneticAlgorithm(
        cfg,
        fitness_fn=fitness_fn,
        parent_selector=parent_selector,
        replacer=replacer,
    )

    history = ga.evolve(pop, generations=60)
    histories.append(history)
    labels.append(label)

plot_multiple_fitness_histories(histories, labels)